1.Data Loading


In [1]:
2+2

4

In [2]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader('data/ml.pdf')
documents = loader.load()
print("Data Loading Complete")
print("Length of documents : ",len(documents))

C:\Users\DELL\AppData\Local\Temp\ipykernel_6808\3481224376.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
d:\rag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data Loading Complete
Length of documents :  851


2.Text Splitting

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(documents)
print("Text Splitting completed")
print("Length of chunks : ",len(chunks))

Text Splitting completed
Length of chunks :  2236


In [17]:
def clean_text(text):
    return "".join(
        ch for ch in text
        if not (0xD800 <= ord(ch) <= 0xDFFF)
    )

In [18]:
for chunk in chunks:
    chunk.page_content = clean_text(chunk.page_content)

3.Create Embeddings

In [19]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-MiniLM-L6-v2')
embed = embeddings.embed_documents([chunk.page_content for chunk in chunks])
print("Embeddings Completed")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2218.49it/s]


Embeddings Completed


4.Vectorstore

In [21]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    collection_name = "data",
    documents = chunks,
    embedding = embeddings,
    persist_directory = "chroma_db"
)
print("Vector Store Created")


Vector Store Created


5.Create a retriever

In [22]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

print("Retriever Created")

Retriever Created


In [23]:
query = "What is linear regression?"

docs = retriever.invoke(query)

print(len(docs))
print(docs[0].page_content)

5
derivatives are. If you are unfamiliar with these concepts, please go
through the linear algebra and calculus introductory tutorials avail‐
able as Jupyter notebooks in the online supplemental material. For
those who are truly allergic to mathematics, you should still go
through this chapter and simply skip the equations; hopefully, the
text will be sufficient to help you understand most of the concepts.
Linear Regression
In Chapter 1 we looked at a simple regression model of life satisfaction: life_satisfac‐
tion = θ0 + θ1 × GDP_per_capita.
This model is just a linear function of the input feature GDP_per_capita. θ0 and θ1 are
the model’s parameters.
More generally, a linear model makes a prediction by simply computing a weighted
sum of the input features, plus a constant called the bias term (also called the intercept
term), as shown in Equation 4-1.
Equation 4-1. Linear Regression model prediction
y = θ0 + θ1x1 + θ2x2 + ⋯ + θnxn
In this equation:
• ŷ is the predicted value.


6.Connect LLM

In [25]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
load_dotenv()
llm = ChatGroq(
    model_name = "llama-3.3-70b-versatile",
    api_key = os.environ.get("GROQ_API_KEY")
)
print("LLM is created")

LLM is created


8.Create a Prompt


In [38]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a friendly and knowledgeable Machine Learning Tutor Bot.

You help students learn ML concepts using only the retrieved context from the study PDF.

Instructions:
- Answer the question only from the provided context.
- Explain in simple and easy-to-understand language.
- Teach like a tutor, not like a search engine.
- When possible, include:
  1. a short definition,
  2. intuition,
  3. a simple example.
- If the context does not contain enough information, say:
  "I could not find enough information in the provided study material."
- Do not hallucinate or invent content.
- Keep the answer well-structured and student-friendly.
- If the student asks for comparison, use bullet points.
- If the student asks for steps or working, explain step by step.

Context:
{context}
"""
    ),
    ("human", "{input}")
])

In [39]:
from langchain_classic.chains.combine_documents import (
    create_stuff_documents_chain,
)
from langchain_classic.chains import create_retrieval_chain

In [40]:
document_chain = create_stuff_documents_chain(
    llm,
    prompt
)

In [41]:
retrieval_chain = create_retrieval_chain(
    retriever,
    document_chain
)

In [42]:
response = retrieval_chain.invoke(
    {"input": "What is Linear Regression?"}
)

print(response["answer"])

**Definition:** Linear Regression is a type of machine learning model that makes predictions by computing a weighted sum of the input features, plus a constant called the bias term.

**Intuition:** Think of Linear Regression as a line that best fits your data. The model tries to find the optimal line that minimizes the difference between the predicted values and the actual values.

**Simple Example:** Let's say we want to predict life satisfaction based on the GDP per capita. We can use a simple Linear Regression model: life_satisfaction = θ0 + θ1 × GDP_per_capita. Here, θ0 is the bias term, and θ1 is the weight of the GDP_per_capita feature.

In this example, the model will find the optimal values of θ0 and θ1 that best fit the data, so that it can make accurate predictions of life satisfaction for new, unseen data.

**Key Equation:** The Linear Regression model prediction is given by the equation: y = θ0 + θ1x1 + θ2x2 + ⋯ + θnxn, where y is the predicted value, x1, x2, ..., xn are th